Turning Human VAD scores into a long format table:

In [1]:
import pandas as pd

In [7]:
dimensions = ['valence', 'arousal', 'dominance']

rows = []

for dimension in dimensions:
    df = pd.read_excel(f'{dimension}.xlsx')
    # first row -> header, drop all empty columns
    df.columns = df.iloc[0]
    df = df[1:]
    df = df.dropna(axis=1)

    # get the tweet ids
    tweet_ids = df['Tweet ID']
    
    for col_index, col_name in enumerate(df.columns[1:]):
        method = 'rs' if col_index < 6 else 'bws' if col_index < 12 else 'pc'
        for tweet_id, score in zip(tweet_ids, df[col_name]):
            rows.append({
                'tweet_id': tweet_id,
                'dimension': dimension,
                'method': method,
                'annotator': col_name,
                'score': float(score) / 100 if method == 'rs' else float(score)
            })

df = pd.DataFrame(rows)
df.head()



,tweet_id,dimension,method,annotator,score
0,['872197389276385280'],valence,rs,A9,0.73
1,['906447947092647936'],valence,rs,A9,0.25
2,['929386800522645508'],valence,rs,A9,0.78
3,['839145863217889282'],valence,rs,A9,0.00
4,['867703984580169730'],valence,rs,A9,0.35


In [8]:
df.to_csv('human_scores.csv', index=False)

Data inspection:

In [9]:
# inspect if the tweet ids are the same as in llm data:

llm = pd.read_csv('../LLMs/llm_pc_bws_scores.csv')
llm.head()

,tweet_id,dimension,method,model,score
0,['881112328653197312'],valence,pc,gpt,0.847615
1,['889056087009308672'],valence,pc,gpt,0.233316
2,['845724240552824832'],valence,pc,gpt,0.394523
3,['819864483141193728'],valence,pc,gpt,0.187276
4,['915059102874824704'],valence,pc,gpt,0.504192


In [10]:
df_tweet_ids = set(df['tweet_id'])
llm_tweet_ids = set(llm['tweet_id'])

if df_tweet_ids == llm_tweet_ids:
    print("The 'tweet_id' columns in df and llm contain the same 100 values.")
else:
    print("The 'tweet_id' columns in df and llm do not contain the same values.")
    print(f"Values in df but not in llm: {df_tweet_ids - llm_tweet_ids}")
    print(f"Values in llm but not in df: {llm_tweet_ids - df_tweet_ids}")

The 'tweet_id' columns in df and llm contain the same 100 values.
